In [ ]:
import os, shutil, pathlib, json
import numpy as np
import jax
import jax.numpy as jnp
import flax.linen as nn
import matplotlib.pyplot as plt
from absl import logging
import ml_collections

from typing import Any, Callable, Sequence, Optional, List, Tuple

from action_angle_networks import train, analysis
from action_angle_networks.simulation import harmonic_motion_simulation as hsim
from action_angle_networks.configs.harmonic_motion import default as hm_default

%config InlineBackend.figure_format = 'retina'
logging.set_verbosity(logging.INFO)

## G-Symp Net
Daigavane et al, 2022にあるブロック
\begin{equation}
f(x)=C x+W^T \operatorname{diag}(A) \sigma(W x+B)
\end{equation}

In [ ]:
class GSBlock(nn.Module):
    dim_config: int # dimenion of configuration space
    dim_hidden: int # hidden dimension of MLP
    activation: Callable = nn.relu 

    def setup(self):

        self.W_affine = nn.Dense(self.dim_hidden, name="W_affine")
        self.A_diag = self.param("A_diag", nn.initializers.ones, (self.dim_hidden,), name="A_diag")
        self.C_lin = nn.Dense(self.dim_config, use_bias=False, name="C_lin")

    @nn.compact
    def __call__(self, x):
        # (B, dim_config) -> (B, dim_hidden)
        y = self.activation(self.W_affine(x))

        # W_affine の kernel を取得（形状は (dim_hidden, dim_config) ※Flax Dense の慣習）
        # z = (A ⊙ y) @ W   だが kernel は (d, m) なので転置して (m, d)
        W_kernel = self.variables["params"]["W_affine"]["kernel"]  # (dim_hidden, dim_config)
        z = (y * self.A_diag) @ W_kernel.T # (B, dim_config)

        return z + self.C_lin(x) # (B, dim_config)

class GSympNet(nn.Module):
    dim_config: int # dimenion of configuration space
    dim_hidden: int # hidden dimension of MLP
    num_blocks: int
    activation: Callable = nn.relu

    assert num_blocks % 2 == 0, "num_blocks must be even number"

    def setup(self):
        self.blocks = List[nn.Module] =[
            GSBlock(
                dim_config=self.dim_config,
                dim_hidden=self.dim_hidden,
                name=f'gsblock_{i}'
                ) for i in range(self.num_blocks)
        ]

    def __call__(self, q, p):
        for idx, blk in enumerate(self.blocks):
            if idx % 2 == 0:
                p = p + blk(q)
            else:
                q = q + blk(p)
        return q, p # I^x, I^y
    
    # 逆写像：層を逆順に、加算の符号を反転（可逆三角写像の基本）
    def inverse(self, q, p):
        for idx in reversed(range(self.num_blocks)):
            blk = self.blocks[idx]
            if idx % 2 == 0:           # p' = p + f(q)  =>  p = p' - f(q)
                p = p - blk(q)
            else:                    # q' = q + f(p)  =>  q = q' - f(p)
                q = q - blk(p)
        return q, p

class PolarCoordinates(nn.Module):
    dim_config: int # dimenion of configuration space

    def setup(self):
        pass

    def __call__(self, Ix, Iy):
        I = jnp.sqrt(Ix**2 + Iy**2)
        theta = jnp.arctan2(Iy, Ix)
        return I, theta

class GotosCanonicalPolarCoordinates(nn.Module):
    dim_config: int # dimenion of configuration space

    def setup(self):
        pass

    def __call__(self, Ix, Iy):
        I = 0.5 * (Ix**2 + Iy**2)
        theta = jnp.arctan2(-Iy, Ix)
        return I, theta

class InversePolarCoordinates(nn.Module):
    dim_config: int # dimenion of configuration space

    def setup(self):
        pass

    def __call__(self, I, theta):
        Ix = jnp.sqrt(I) * jnp.cos(theta)
        Iy = jnp.sqrt(I) * jnp.sin(theta)
        return Ix, Iy

class InverseGotosCanonicalPolarCoordinates(nn.Module):
    dim_config: int # dimenion of configuration space

    def setup(self):
        pass

    def __call__(self, I, theta):
        Ix = jnp.sqrt(2*I) * jnp.cos(theta)
        Iy = -jnp.sqrt(2*I) * jnp.sin(theta)
        return Ix, Iy

class MLP(nn.Module):
    dim_input: int
    dim_output: int
    dim_hidden: int

    def setup(self):
        self.dense1 = nn.Dense(self.dim_hidden, activation=nn.relu)
        self.dense2 = nn.Dense(self.dim_output)

    def __call__(self, x):
        x = self.dense1(x)
        x = self.dense2(x)
        return x

class MyActionAngleNetwork(nn.Module):
    dim_config: int # dimenion of configuration space
    dim_hidden: int # hidden dimension of MLP
    num_gsblocks: int
    type_polar: str = "canonical" # "canonical" or "normal"
    activation: Callable = nn.relu

    def setup(self):
        self.gsymp_net = GSympNet(
            dim_config=self.dim_config,
            dim_hidden=self.dim_hidden,
            num_blocks=self.num_gsblocks,
            activation=self.activation
        )
        if self.type_polar == "normal":
            self.to_polar = PolarCoordinates(dim_config=self.dim_config)
            self.inv_polar = InversePolarCoordinates(dim_config=self.dim_config)
        elif self.type_polar == "canonical":
            self.to_polar = GotosCanonicalPolarCoordinates(dim_config=self.dim_config)
            self.inv_polar = InverseGotosCanonicalPolarCoordinates(dim_config=self.dim_config)
        self.theta_generator = MLP(
            dim_input=1,
            dim_output=1,
            dim_hidden=self.dim_hidden
        )
        # self.to_polar = PolarCoordinates(dim_config=self.dim_config)
        # self.to_canonical_polar = GotosCanonicalPolarCoordinates(dim_config=self.dim_config)
        # self.inv_to_polar = InversePolarCoordinates(dim_config=self.dim_config)
        # self.inv_to_canonical_polar = InverseGotosCanonicalPolarCoordinates(dim_config=self.dim_config)

    def __call__(self, q, p, delta_t):
        # (q, p) -> (Ix, Iy)
        Ix, Iy = self.gsymp_net(q, p)

        # (Ix, Iy) -> (I, theta)
        I, theta = self.to_polar(Ix, Iy)

        theta_ = theta + delta_t * self.theta_generator(I)

        Ix_, Iy_ = self.inv_polar(I, theta_)
        q_, p_ = self.gsymp_net.inverse(Ix_, Iy_)

        return q_, p_

# class ThetaEvolver(nn.Module):

#     def setup(self):
#         self.generator = MLP(
#             dim_input=,
#             dim_output=,
#             dim_hidden=
#         )

#     def __call__(self, I0):

        


#         return self.generator(x)


In [ ]:


@jit
def step(params, opt_state, x, y, key):
    def loss_fn(p):
        Ix, Iy = model.apply({'params': p}, x, train=True, rngs={'noise': key})
        



        logits = model.apply({'params': p}, x, train=True, rngs={'noise': key})
        onehot = jax.nn.one_hot(y, 10)
        return optax.softmax_cross_entropy(logits, onehot).mean()
    loss, grads = value_and_grad(loss_fn)(params)
    updates, opt_state = tx.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

@jit
def eval_batch(params, x, y, key):
    logits = model.apply({'params': params}, x, train=False,  rngs={'noise': key})
    pred = logits.argmax(axis=-1)
    acc = (pred == y).mean()
    return acc

# ===== 学習ループ =====

def train_step():


    
